In [ ]:
#!pip install scikeras==0.13.0 scikit-learn==1.5.2

In [5]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import GridSearchCV
import kagglehub
from tensorflow.keras.models import Sequential
from scikeras.wrappers import KerasClassifier

In [6]:
df =pd.read_csv('/content/sample_data/diabetes.csv')
# carrega o dataset

In [7]:
df.head()
# verifica primeiras linhas

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [8]:
df.isnull().sum()
# verifica se há necessidade de limpeza

,0
Pregnancies,0
Glucose,0
BloodPressure,0
SkinThickness,0
Insulin,0
BMI,0
DiabetesPedigreeFunction,0
Age,0
Outcome,0


In [9]:
x = df[['Pregnancies','Glucose','BloodPressure','SkinThickness','Insulin','BMI','DiabetesPedigreeFunction','Age']]
y = df['Outcome']
# separa as features previsoras e a target

In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)

In [11]:
y

,Outcome
0,1
1,0
2,1
3,0
4,1
...,...
763,0
764,0
765,0
766,1


In [12]:
def rede (optimizer, loss, initializer, activation, neurons):
    # recebe os parametros do modelos para cada vez que mudar os parametros no grid search
    rede = Sequential([
        tf.keras.layers.InputLayer(shape=(8,)),
        # 8 features 8 neuronios de entrada
        tf.keras.layers.Dense(units=neurons, activation=activation, kernel_initializer=initializer),
        # camada oculta
        tf.keras.layers.Dropout(rate=0.2),
        # ajuda a minimizar o overfitting
        tf.keras.layers.Dense(units=neurons, activation=activation, kernel_initializer=initializer),
        # camada oculta
        tf.keras.layers.Dropout(rate=0.2),
        tf.keras.layers.Dense(units=1,activation='sigmoid')
        # um neuronio de saida
    ])
    rede.compile(optimizer=optimizer,loss=loss, metrics = ['binary_accuracy'])
    return rede

In [13]:
rede = KerasClassifier(model= rede)

In [14]:
params = {
    'batch_size':[10],
    'epochs':[50],
    'model__optimizer': ['adam','sgd'],
    'model__loss': ['binary_crossentropy'],
    'model__initializer': ['normal','uniform'],
    'model__activation': ['relu'],
    'model__neurons': [4,6]
}
# em um dict define todos os parametros que serao testados

In [15]:
gs = GridSearchCV(estimator=rede,param_grid=params, scoring='accuracy', cv=4)
# cria o modelo do grid search, com os parametros definidos no dict, cv significa o cross validation

In [16]:
gs = gs.fit(x_scaled,y)
# treina o modelo com diferentes parametros

Epoch 1/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - binary_accuracy: 0.6458 - loss: 0.6846
Epoch 2/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.6510 - loss: 0.6601
Epoch 3/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.6510 - loss: 0.6255
Epoch 4/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.6510 - loss: 0.5907
Epoch 5/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.6493 - loss: 0.5701
Epoch 6/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.6771 - loss: 0.5550
Epoch 7/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.7413 - loss: 0.5289
Epoch 8/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.7274 - loss: 0.5188
Epoch 9/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.7448 - loss: 0.5152
Epoch 10/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.7274 - loss: 0.5156
Epoch 11/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.7222 - loss: 0.52

In [ ]:
best_params = gs.best_params_
# prga os melhores parametros
best_acc = gs.best_score_
#pega os melhores parametros

In [18]:
print(best_params)
print(best_acc)

{'batch_size': 10, 'epochs': 50, 'model__activation': 'relu', 'model__initializer': 'normal', 'model__loss': 'binary_crossentropy', 'model__neurons': 4, 'model__optimizer': 'adam'}
0.7669270833333334


In [ ]:
model =gs.best_estimator_
# melhor modelo

In [ ]:
predict = model.predict([[6,148,72,35,0	,33.6,0.627,50]])
#predição simples
predict

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step


array([1])

In [ ]:
model.model_.save('modelo_diabetes.keras')
#salva o modelo

In [ ]:
modelo_carregado = tf.keras.models.load_model('modelo_diabetes.keras')
# carrega o model salvo
dado_novo = [[65, 138, 32, 35, 0, 37.6, 0.627, 24]]
# lista para uma nova predição
dado_novo_scaled = scaler.transform(dado_novo)
# como eu usei o transformer, para inserir e necessario transforma-los
predict = modelo_carregado.predict(dado_novo_scaled)

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 319ms/step


In [29]:
predict

array([[0.94792545]], dtype=float32)

In [30]:
if predict >0.8 :
  predict=True
else:
  predict=False

In [31]:
predict

True